In [1]:
import pandas as pd
import numpy as np

roll_number = "1024170251"
categories = ["billing", "account", "general"]

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee",
     "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

last_two_digits = [int(d) for d in roll_number[-2:]]
personalized_entries = []

for digit in last_two_digits:
    category = categories[digit % 3]

    if category == "billing":
        entry = {
            "question": "how can i check my fee payment status",
            "answer": "You can check your fee payment status in the billing section.",
            "keywords": "fee payment status billing",
            "category": "billing"
        }
    elif category == "account":
        entry = {
            "question": "how do i update my registered mobile number",
            "answer": "Go to Account Settings and update your registered mobile number.",
            "keywords": "mobile number update account",
            "category": "account"
        }
    else:
        entry = {
            "question": "where can i find general campus information",
            "answer": "You can find general campus information on the student portal.",
            "keywords": "campus information general portal",
            "category": "general"
        }

    personalized_entries.append(entry)

faq_df = pd.DataFrame(fixed_entries + personalized_entries)
print(faq_df.to_string(index=False))


                                   question                                                           answer                          keywords category
                     what is the annual fee                                        The annual fee is Rs 500.             fee cost price charge  billing
                      how to reset password                                 Go to Settings > Reset Password.              password reset login  account
                what are your working hours                                        We are open 9 AM to 5 PM.            hours timing open time  general
                      how can i pay the fee                       You can pay via UPI, card, or net banking.               pay payment upi fee  billing
where can i find general campus information   You can find general campus information on the student portal. campus information general portal  general
how do i update my registered mobile number Go to Account Settings and update your regis

In [2]:
def score_query(query, df):
    query_words = set(query.lower().split())
    scores = []

    for _, row in df.iterrows():
        text = (row["question"] + " " + row["keywords"]).lower()
        text_words = set(text.split())
        score = len(query_words & text_words)
        scores.append(score)

    result = df.copy()
    result["confidence"] = scores

    return result[result["confidence"] > 0].sort_values(
        by="confidence", ascending=False
    )

query = "how to pay fee"
result = score_query(query, faq_df)

print(result.to_string(index=False))


                                   question                                                           answer                     keywords category  confidence
                      how can i pay the fee                       You can pay via UPI, card, or net banking.          pay payment upi fee  billing           3
                      how to reset password                                 Go to Settings > Reset Password.         password reset login  account           2
                     what is the annual fee                                        The annual fee is Rs 500.        fee cost price charge  billing           1
how do i update my registered mobile number Go to Account Settings and update your registered mobile number. mobile number update account  account           1


In [3]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]

category_name = faq_df.iloc[4]["category"]
print(same_category(category_name, faq_df).to_string(index=False))


                                   question                                                         answer                          keywords category
                what are your working hours                                      We are open 9 AM to 5 PM.            hours timing open time  general
where can i find general campus information You can find general campus information on the student portal. campus information general portal  general


In [4]:
selected_index = 0

new_keyword = input("Enter a new keyword: ")
faq_df.loc[selected_index, "keywords"] += " " + new_keyword

csv_filename = f"{roll_number}_faq_data.csv"
faq_df.to_csv(csv_filename, index=False)

print(faq_df.loc[selected_index].to_string())
print(f"Saved as {csv_filename}")


Enter a new keyword: question           what is the annual fee
answer          The annual fee is Rs 500.
keywords    fee cost price charge support
category                          billing
Saved as 1024170251_faq_data.csv


In [5]:
print(faq_df.groupby("category").size())


category
account    2
billing    2
general    2
dtype: int64


In [6]:
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())
    scores = []

    for _, row in df.iterrows():
        text = (row["question"] + " " + row["keywords"]).lower()
        text_words = set(text.split())
        score = len(query_words & text_words)
        scores.append(score)

    result = df.copy()
    result["confidence"] = scores
    result = result[result["confidence"] > 0].sort_values(
        by="confidence", ascending=False
    )

    if result.empty:
        print("No matching entries found.")
        return result

    highest_score = result["confidence"].max()
    highest_matches = result[result["confidence"] == highest_score]

    if len(highest_matches) > 1:
        print("Tie detected. All highest-scoring entries:")
        print(highest_matches.to_string(index=False))
    else:
        print("Best matching entry:")
        print(highest_matches.to_string(index=False))

    return result

print("Tie query:")
score_query_with_ties("fee", faq_df)

print("\nNon-tie query:")
score_query_with_ties("password", faq_df)


Tie query:
Tie detected. All highest-scoring entries:
              question                                     answer                      keywords category  confidence
what is the annual fee                  The annual fee is Rs 500. fee cost price charge support  billing           1
 how can i pay the fee You can pay via UPI, card, or net banking.           pay payment upi fee  billing           1

Non-tie query:
Best matching entry:
             question                           answer             keywords category  confidence
how to reset password Go to Settings > Reset Password. password reset login  account           1
